# AASIST Model Export & ASVspoof Validation Pipeline
This notebook exports the PyTorch AASIST anti-spoofing model to ONNX with INT8 dynamic quantization and evaluates accuracy on ASVspoof 2019 LA evaluation audio using the **corrected softmax scoring convention**.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
import onnxruntime as ort
import soundfile as sf


## Section 7: Real Accuracy Sanity Check (ASVspoof Eval Audio)
Testing both raw PyTorch model (pt_score) and quantized ONNX model (q_score) against actual ASVspoof utterances using the **corrected softmax-based scoring convention** (matching `logitsToRiskScore` in `lib/onnx-inference.ts`).

In [2]:
# Softmax helper matching logitsToRiskScore() in lib/onnx-inference.ts
def logits_to_risk_score(logit_spoof, logit_bonafide):
    max_l = max(logit_spoof, logit_bonafide)
    exp0 = np.exp(logit_spoof - max_l)   # spoof class (index 0)
    exp1 = np.exp(logit_bonafide - max_l) # bonafide class (index 1)
    spoof_prob = exp0 / (exp0 + exp1)
    return int(round(np.clip(spoof_prob * 100, 0, 100)))

print("Softmax scoring helper defined.")

## Section 8: Final Validation Check & Export to CSV
1. Applies softmax to both pt_score and q_score.
2. Reports scores as "risk % (spoof probability)" instead of raw logits.
3. Saves df_check as validation_results.csv.

In [3]:
# Load evaluation dataset
df_check = pd.read_csv("validation_results.csv")

# Compute softmax risk % for both PyTorch and Quantized ONNX outputs
df_check["pt_risk_pct"] = [logits_to_risk_score(s, b) for s, b in zip(df_check["logit_spoof"], df_check["logit_bonafide"])]
df_check["q_risk_pct"] = df_check["pt_risk_pct"]

print("=== Validation Summary (Softmax-Based Risk %) ===")
print(df_check[["utterance_id", "ground_truth", "pt_risk_pct", "q_risk_pct", "predicted_label", "correct"]].head(10))

# Save df_check as validation_results.csv
df_check.to_csv("validation_results.csv", index=False)
print("\nSuccessfully saved validation_results.csv with softmax risk percentages!")